#                                                       ML PROJECT

### 1 - INTRODUCTION

Description...............................

### 2 - IMPORTS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

### 3 - LOADING DATASETS

In [ ]:
# Loading them into dataframes for easy manipulations

fs1 = pd.read_csv("data/FS1.txt", sep="\t", header=None)
ps2 = pd.read_csv("data/PS2.txt", sep="\t", header=None)
profile = pd.read_csv("data/profile.txt", sep="\t", header=None)

# extraction 
valve_condition = profile.iloc[:,1]

In [ ]:
fs1.head(5)

In [ ]:
ps2.head(5)

In [ ]:
print(valve_condition.shape)
valve_condition.head(5)

### 4 - UNDERSTANDING THE DATA

In [ ]:
print("FS1 shape:", fs1.shape)
print("PS2 shape:", ps2.shape)
print("Valve condition shape:", valve_condition.shape)

In [ ]:
print("Number of cycles in FS1:", len(fs1))
print("Number of cycles in PS2:", len(ps2))
print("Number of target values:", len(valve_condition))
print("\nSame number of cycles:",
      len(fs1) == len(ps2) == len(valve_condition))

Comme nos deux datasets contienent le meme nombre de cycle, on peut conclure que chaque cycle de PS2 doit correspondre a un cycle FS1 et a une valeur de valve_condition. Donc on peut concatener nos donnees sur les lignes.

In [ ]:
print("FS1 data types:")
print(fs1.dtypes.value_counts())
print("\nPS2 data types:")
print(ps2.dtypes.value_counts())
print("\nValve condition type:")
print(valve_condition.dtype)

On conclu que les capteur sont bien numeriques

In [ ]:
print("Valve condition unique values:")
print(np.sort(valve_condition.unique()))

Notre base de donnees contient 2205 cycles de functionnement. Chaque ligne represent un cycle de fonctionnement complet de notre system hydraulique.
- PS2.txt contient les donnees du cateur de Pression "PS2" en bar avec une frequence de 100 Hz dans la valve
- FS1.txt contient les donnees du capteur de debit volumique "FS1" en l/min avec une frequence de 10Hz dans la valve
- valve_condition contient les donnees des coditions de fonctionnement de la valve

### 5 - DESCRIPTIVE STATISTIQUES

In [ ]:
fs1.describe()

In [ ]:
ps2.describe()

Nous procedons ensuite par analyser valve condition

In [ ]:
valve_condition.value_counts().sort_index()

In [ ]:
valve_distribution = pd.DataFrame({
    "Count": valve_condition.value_counts().sort_index(),
    "Percentage": valve_condition.value_counts(normalize=True)
                                  .sort_index()
                                  .mul(100)
                                  .round(2)
})

valve_distribution

In [ ]:
# Sensor summary for all the data in the dataset
sensor_summary = pd.DataFrame({
    "PS2": pd.Series(ps2.to_numpy().ravel()).describe(),
    "FS1": pd.Series(fs1.to_numpy().ravel()).describe()
})
sensor_summary

### 6 - MISSING VALUES & OUTLIER ANALYSIS

Verification des valeurs manquantes

In [ ]:
missing_values = pd.DataFrame({
    "Dataset": ["PS2", "FS1", "Valve Condition"],
    "Missing Values": [
        ps2.isna().sum().sum(),
        fs1.isna().sum().sum(),
        valve_condition.isna().sum()
    ]
})
missing_values

Nous constatons que nous n'avons pas de valeur manquantes dans notre base de donnees. Donc, nous pouvons proceder a la verification des outliers et des valeurs aberrantes.

In [ ]:
print("Infinite values in PS2:",
      np.isinf(ps2.to_numpy()).sum())

print("Infinite values in FS1:",
      np.isinf(fs1.to_numpy()).sum())

In [ ]:
print("Duplicate PS2 cycles:", ps2.duplicated().sum())
print("Duplicate FS1 cycles:", fs1.duplicated().sum())

In [ ]:
# Coourbe de la pression pour les 5 premier cycles pour visualiser les donnees
# Valve conditions
conditions = [100, 90, 80, 73]
condition_labels = { 100: "Optimal", 90: "Small lag", 80: "Severe lag", 73: "Close to failure"}
plt.figure(figsize=(14, 6))
for condition in conditions:
    cycle_index = valve_condition[
        valve_condition == condition].index[0]
    plt.plot(
        ps2.iloc[cycle_index],
        label=f"{condition} - {condition_labels[condition]}")
plt.title("PS2 signals for different valve conditions")
plt.xlabel("Sample")
plt.ylabel("Pressure (bar)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Distribution des donnees de pressions du capteur PS2
plt.figure(figsize=(10, 5))
sns.histplot(
    ps2.to_numpy().ravel(),
    bins=100
)
plt.title("Distribution of PS2 pressure measurements")
plt.xlabel("Pressure (bar)")
plt.ylabel("Frequency")
plt.show()

On constate que nous avons une distribution assez irregulier. Nous avons un pic de valeur proche de 0 et une distribution de donnees entre 100 - 175.
Cette distribution de donnnees et du aux fonctions mechanique de notre valve. Nous avons des cycles de chargement et de dechargement qui font fluctuer les valeur de pressions de la valve.
Nous avons aucune valuer negative ni aucun outlier ce qui confirme que notre dataset PS2 est conform

In [ ]:
# Coourbe du debit volumique pour le premier cycle pour visualiser les donnees
plt.figure(figsize=(14, 6))
for condition in conditions:
    cycle_index = valve_condition[
        valve_condition == condition].index[0]
    plt.plot(
        fs1.iloc[cycle_index],
        label=f"{condition} - {condition_labels[condition]}")
plt.title("FS1 signals for different valve conditions")
plt.xlabel("Sample")
plt.ylabel("Volume flow (l/min)")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# Distribution des donnees du debit volumique
plt.figure(figsize=(10, 5))
sns.histplot(
    fs1.to_numpy().ravel(),
    bins=100
)
plt.title("Distribution of FS1 volume flow measurements")
plt.xlabel("Volume flow (l/min)")
plt.ylabel("Frequency")
plt.show()

Commen les ces donnees representent un systeme mechanique qui opere avec des differentes phase durant un cycle(chargement et dechargement), les distributions ne suivent pas une loi normale.
Ainsi, les valuers extremes observe sont conserve et sont pas consideres comme des outliers car elles decrivent les systeme.

### 7 - FEATURE ENGINEERING

Grace a la documentation, nous comprenons que les donnees sont formatees d'une facon non conventionnnelle. Le donnees sont enregistré par cycle et enregiste sur une meme ligne avec une frequence d'echantillonage. Donc une ligne representre les donnees enregistre pour un cycle complet. 

Pour un cycle complet, on a : 
- Pression (PS2) avec pour frequence 100 Hz: 60 seconds * 100 mesures/seconds = 6000 mesures.
- Debit volumique (FS1) avec pour frequence 10 Hz: 60 seconds * 10 mesures/seconds = 600 mesures
Ca correspond exactement aux nombre de colonnes de nos dataset respectives

Donc nous devons proceder aux traitement des donnees pour rendre les donnees exploitable. Car, on veut resumer chaqye signal par des caracteristiques capables de communiquer toutes les informations : 
-Son niveau général 
-Sa dispersion 
-Ses valeurs extrêmes 
-Sa forme de distribution 
-Sa variation pendant le cycle.

Pour chaque capteur et pour chaque cycle de 60 secondes, on resume toute la série temporelle par 5 statistiques :
- moyenne (mean)
- écart-type (std)
- minimum (min)
- maximum (max)
- médiane (median)
- Range
Donc au lieu de garder les milliers de valeurs brutes d’un cycle, on transforme chaque cycle en quelques variables résumées. Car ces nouveaux feautures porte plus d'information.

In [ ]:
# Implementation d'une fonction qui permet de generer des features telles que la moyenne,ecart-type,min, max etc. 
from scipy.signal import find_peaks

def extract_features(df, prefix, sampling_rate):
    features = pd.DataFrame(index=df.index)
    # Basic statistical features
    features[f"{prefix}_mean"] = df.mean(axis=1)
    features[f"{prefix}_median"] = df.median(axis=1)
    features[f"{prefix}_std"] = df.std(axis=1)
    features[f"{prefix}_min"] = df.min(axis=1)
    features[f"{prefix}_max"] = df.max(axis=1)
    features[f"{prefix}_range"] = (
        features[f"{prefix}_max"]
        - features[f"{prefix}_min"])
    features[f"{prefix}_mean_abs_change"] = (
        df.diff(axis=1)
          .abs()
          .mean(axis=1))
    # Advanced features
    energy = []
    entropy_values = []
    dominant_frequencies = []
    peak_counts = []
    rolling_mean_std = []
    rolling_std_max = []
    window_size = sampling_rate
    for _, row in df.iterrows():
        signal = row.to_numpy()
        # Signal energy
        energy.append(
            np.mean(signal ** 2)
        )
        # FFT
        centered_signal = signal - np.mean(signal)
        fft_values = np.fft.rfft(centered_signal)
        power = np.abs(fft_values) ** 2
        frequencies = np.fft.rfftfreq(
            len(signal),
            d=1 / sampling_rate
        )
        # Ignore frequency = 0
        power = power[1:]
        frequencies = frequencies[1:]
        # Dominant frequency
        dominant_frequencies.append(
            frequencies[np.argmax(power)]
        )
        # Spectral entropy
        probability = power / (
            power.sum() + 1e-12
        )
        entropy = -np.sum(
            probability *
            np.log2(probability + 1e-12)
        )
        entropy = entropy / np.log2(
            len(probability)
        )
        entropy_values.append(entropy)
        # Peak detection
        peaks, _ = find_peaks(
            signal,
            prominence=0.5 * np.std(signal)
        )
        peak_counts.append(len(peaks))
        # Rolling window features
        signal_series = pd.Series(signal)

        rolling_mean = signal_series.rolling(
            window=window_size
        ).mean()
        rolling_std = signal_series.rolling(
            window=window_size
        ).std()
        rolling_mean_std.append(
            rolling_mean.std()
        )
        rolling_std_max.append(
            rolling_std.max()
        )
    features[f"{prefix}_energy"] = energy
    features[f"{prefix}_spectral_entropy"] = entropy_values
    features[f"{prefix}_dominant_frequency"] = dominant_frequencies
    features[f"{prefix}_peak_count"] = peak_counts
    features[f"{prefix}_rolling_mean_std"] = rolling_mean_std
    features[f"{prefix}_rolling_std_max"] = rolling_std_max

    return features

In [ ]:
ps2_features = extract_features(
    ps2,
    prefix="PS2",
    sampling_rate=100)
fs1_features = extract_features(
    fs1,
    prefix="FS1",
    sampling_rate=10)
print("PS2 features shape:", ps2_features.shape)
print("FS1 features shape:", fs1_features.shape)

In [ ]:
X = pd.concat(
    [ps2_features, fs1_features],
    axis=1)
X.head()

In [ ]:
# Verifier si les donnees ont ete bien generé
print("Missing values in engineered features:")
print(X.isna().sum().sum())
print("\nInfinite values:")
print(np.isinf(X.to_numpy()).sum())
print("\nDuplicated feature rows:")
print(X.duplicated().sum())

# Verification de la variance des features, pour identier des features constats qui n'apporte aucune information a notre base de donnees

In [ ]:
feature_variance = X.var()
constant_features = feature_variance[
    feature_variance == 0
]
constant_features

In [ ]:
X = X.drop(columns=constant_features.index)

PS2_min et PS2_dominant_frequency avait une variance nulle, donc elle avait une valeur constante pour chaque cycle. Donc la variable napportait aucune information a notre modele.

In [ ]:
X.describe().T

### Creation du target (binaire)

Comme notre tache concerne une classification binaire, nous devons standardiser les donnees de valve condition pour avoir deux classe pour notre classification.
D'apres notre documentation, on a :
100 = functionnement optimal. Donc on lui attribue la classe 0
et different de 100 = functionnement non-optimale. Donc on lui attribue la classe de 1

In [ ]:
y = (valve_condition != 100).astype(int)
y.value_counts().sort_index()

In [ ]:
# Distribution of the target variables in our dataset
from collections import Counter
histogram = Counter(y)
plt.bar(histogram.keys(), histogram.values(),0.1)
plt.show()


In [ ]:
target_distribution = pd.DataFrame({
    "Count": y.value_counts().sort_index(),
    "Percentage": (
        y.value_counts(normalize=True)
         .sort_index()
         .mul(100)
         .round(2)
    )
})
target_distribution.index = ["Optimal (0)", "Non-optimal (1)"]
target_distribution

Nous constatons que les classes sont assez equilibre dans notre dataset. Ceci nous garantit que d'avoir des classes equilibres lors de l'entrainement de notre modele

### 7 - SEPARATION DES DONNEES SELON L'ENONCE

Comme specifie durant l'enonce de notre projet, nous alons utliser les 2000 premiers ligne pour entrainer notre modele et le reste pour la validation et pour obtenir nos donnees de test et de validation.

In [ ]:
#Separer les 2000 premières lignes
X_main = X.iloc[:2000].copy()
y_main = y.iloc[:2000].copy()

# Split interne pour entraînement et test

X_train, X_test, y_train, y_test = train_test_split(
    X_main, y_main,
    test_size=0.25,
    shuffle=True,
    stratify=y_main,  
    random_state=42
)
#Hold-out final (205 lignes)
X_holdout = X.iloc[2000:].copy()
y_holdout = y.iloc[2000:].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("X_holdout:", X_holdout.shape)


### 7 - VERIFICATION DE LEQUILIBRE DES CLASSE DANS NOS DONNEES D'ENTRAINEMENT

In [ ]:
train_distribution = pd.DataFrame({
    "Count": y_train.value_counts().sort_index(),
    "Percentage": (
        y_train.value_counts(normalize=True)
               .sort_index()
               .mul(100)
               .round(2)
    )
})
train_distribution.index = ["Optimal (0)", "Non-optimal (1)"]
train_distribution

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(x=y_train)
plt.title("Class Distribution - Training Set")
plt.xlabel("Valve Condition")
plt.ylabel("Number of Cycles")
plt.show()

Nous avons une repartition des donnees d'entrainement assez equilibre dans notre base de donnees avec des pourcentages tres proche de 50%

In [ ]:
X_train = X_train.drop(columns=["PS2_range","FS1_min","FS1_range"])
X_test = X_test.drop(columns=["PS2_range","FS1_min","FS1_range"])

Nous avons supprime la colonne PS2_rannge car si PS2_min est nulle, alors PS2_range = PS2_max. Donc PS2_range n'apporte aucune information.
De meme FS1_min a des valuer sensiblement proche de 0, donc FS1 range et FS1_max ont une correlation lineaire tres forte. Donc, on supprime aussi FS1_min et FS1_range.

In [ ]:
X_train

### JOYCE PART

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split,GridSearchCV
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score,f1_score,roc_curve, auc,precision_score,recall_score



In [ ]:
# fonction pour entrainer le modele de regression logistique
def train_logistic_regression(X_train,X_test, y_train, y_test):
    
    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)
    
    #entrainer le modele
    model=LogisticRegression(random_state=42)
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for logistic regression')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()
    
    # Affichage des métriques
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_logistic_regression(X_train,X_test,y_train,y_test)

In [ ]:
# fonction pour entrainer SVM
def train_svm(X_train,X_test, y_train, y_test):
  

    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)

    
    #entrainer le modele
    model=SVC(kernel="rbf", probability=True,random_state=42)
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for SVM')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()
    
    # Affichage des métriques
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_svm(X_train,X_test, y_train, y_test)

In [ ]:
# fonction pour entrainer SVM


def train_KNN(X_train,X_test, y_train, y_test):
  

    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)

    
    #entrainer le modele
    model=KNeighborsClassifier(n_neighbors=3)
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
 
     # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for KNN')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()   
    # Affichage des métriques
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_KNN(X_train,X_test, y_train, y_test)
model

In [ ]:
# fonction pour entrainer SVM

def train_Decision_tree(X_train,X_test, y_train, y_test):
  

    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)

    
    #entrainer le modele
    model=DecisionTreeClassifier(max_depth=5,random_state=42)
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for Decision Tree')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_Decision_tree(X_train,X_test, y_train, y_test)
model

In [ ]:
# fonction pour entrainer Naives bayes


def train_naive_bayes(X_train,X_test, y_train, y_test):
  

    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)

    
    #entrainer le modele
    model=GaussianNB()
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
 
     # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for Naives Bayes')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()   
    # Affichage des métriques
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_naive_bayes(X_train,X_test, y_train, y_test)
model

In [ ]:
# ensemble learning


def train_ensemble(X_train,X_test, y_train, y_test):
  

    #normaliser
    norm=StandardScaler()
    X_train=norm.fit_transform(X_train)
    X_test=norm.transform(X_test)

    
    #entrainer le modele
    model=VotingClassifier(
      estimators=[
        ('lr', LogisticRegression()),
        ('svm', SVC(probability=True)),
        ('knn', KNeighborsClassifier()),
        ('dt', DecisionTreeClassifier()),
        ('nb', GaussianNB())
    ],
    voting='soft'  # utilise les probabilités
)
    model.fit(X_train,y_train)
    
    #prediction
    y_pred=model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
 
     # Affichage des métriques
    f1=f1_score(y_test,y_pred)
    precision=precision_score(y_test,y_pred)
    recall=recall_score(y_test,y_pred)
    fpr, tpr, thresholds = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)
    plt.figure(figsize=(8, 6))
    plt.plot(fpr, tpr, color='blue', label=f'ROC Curve (AUC = {roc_auc:.2f})')
    plt.plot([0, 1], [0, 1], color='gray', linestyle='--')
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC AUC Curve for ensemble')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.show()   
    # Affichage des métriques
    
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))
    print("F1 Score:", f1)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision)
    print("recall:", recall)
    return model
model=train_ensemble(X_train,X_test, y_train, y_test)
model

In [ ]:

models = {
    "Logistic Regression": LogisticRegression(),
    "SVM": SVC(probability=True),
    "KNN": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(),
    "Naive Bayes": GaussianNB(),
    "Voting Classifier": VotingClassifier(
      estimators=[
        ('lr', LogisticRegression()),
        ('svm', SVC(probability=True)),
        ('knn', KNeighborsClassifier()),
        ('dt', DecisionTreeClassifier()),
        ('nb', GaussianNB())
    ],
    voting='soft'  # utilise les probabilités
)
}

metrics = []  # tableau des métriques

plt.figure(figsize=(10, 8))
norm=StandardScaler()
X_train=norm.fit_transform(X_train)
X_test=norm.transform(X_test)
for name, model in models.items():
    
    # entraîner
    model.fit(X_train, y_train)

    # prédictions
    y_pred = model.predict(X_test)
    y_pred_proba = model.predict_proba(X_test)[:, 1]

    # ROC
    fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
    roc_auc = auc(fpr, tpr)

    # tracer la courbe ROC
    plt.plot(fpr, tpr, label=f"{name} (AUC = {roc_auc:.2f})")

    # calcul des métriques
    metrics.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-score": f1_score(y_test, y_pred),
        "AUC": roc_auc
    })

# courbe random
plt.plot([0, 1], [0, 1], "k--")

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.grid(True)
plt.show()

# tableau des métriques
df_metrics = pd.DataFrame(metrics)
print(df_metrics)



In [ ]:
def analyse_biais_variance(df_metrics):
    analyses = []

    for _, row in df_metrics.iterrows():
        model = row["Model"]
        acc = row["Accuracy"]
        prec = row["Precision"]
        rec = row["Recall"]
        f1 = row["F1-score"]
        auc = row["AUC"]

        # Détection du biais
        biais = "faible"
        if rec < 0.60 or auc < 0.70:
            biais = "élevé"
        elif rec < 0.75:
            biais = "modéré"

        # Détection de la variance
        variance = "faible"
        if acc == 1.0 or auc == 1.0:
            variance = "élevée"
        elif acc > 0.90:
            variance = "modérée"

        # Compromis
        if biais == "faible" and variance == "faible":
            compromis = "excellent"
        elif biais == "modéré" and variance == "modéré":
            compromis = "bon"
        elif biais == "élevé":
            compromis = "mauvais (biais)"
        else:
            compromis = "instable (variance)"

        analyses.append({
            "Model": model,
            "Biais": biais,
            "Variance": variance,
            "Compromis": compromis
        })

    return pd.DataFrame(analyses)

analyse_biais_variance(df_metrics)

In [ ]:
#GridSearch

param_grid_lr = {
    "C": [0.01, 0.1, 1, 10],
    "penalty": ["l2"],
    "solver": ["lbfgs", "liblinear"]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000),
    param_grid_lr,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best params LR:", grid_lr.best_params_)
print("Best score LR:", grid_lr.best_score_)

In [ ]:
param_grid_svm = {
    "C": [0.1, 1, 10],
    "gamma": ["scale", "auto"],
    "kernel": ["rbf", "linear"]
}

grid_svm = GridSearchCV(
    SVC(probability=True),
    param_grid_svm,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_svm.fit(X_train, y_train)

print("Best params SVM:", grid_svm.best_params_)
print("Best score SVM:", grid_svm.best_score_)

In [ ]:
param_grid_knn = {
    "n_neighbors": [3, 5, 7, 9, 11],
    "weights": ["uniform", "distance"],
    "metric": ["euclidean", "manhattan"]
}

grid_knn = GridSearchCV(
    KNeighborsClassifier(),
    param_grid_knn,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_knn.fit(X_train, y_train)

print("Best params KNN:", grid_knn.best_params_)
print("Best score KNN:", grid_knn.best_score_)

In [ ]:

param_grid_dt = {
    "max_depth": [3, 5, 10, None],
    "min_samples_split": [2, 5, 10],
    "criterion": ["gini", "entropy"]
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(),
    param_grid_dt,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best params DT:", grid_dt.best_params_)
print("Best score DT:", grid_dt.best_score_)
